In [2]:
import os
import glob
import time
import gc
import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import timedelta
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from pyproj import Transformer

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })
    return tiles

# ======================== Overlay + GIS-Consistent Info ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(
        results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):

        cls = int(cls.item())
        conf = float(conf.item())
        x1, y1, x2, y2 = map(float, box.tolist())

        mask_pixel_coords = np.array(mask.tolist(), dtype=np.float32)
        mask_gis_coords = []
        for px, py in mask_pixel_coords:
            try:
                lon, lat = rasterio.transform.xy(transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        pixel_count = len(mask_pixel_coords)
        area = pixel_count * (transform.a * abs(transform.e))
        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy
        lon_center, lat_center = rasterio.transform.xy(transform, local_cy, local_cx)
        lon_x1, lat_y1 = rasterio.transform.xy(transform, y1, x1)
        lon_x2, lat_y2 = rasterio.transform.xy(transform, y2, x2)
        bbox_width_geo = abs(lon_x2 - lon_x1)
        bbox_height_geo = abs(lat_y2 - lat_y1)

        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "bbox_lat_y1": lat_y1,
            "bbox_lon_x1": lon_x1,
            "bbox_lat_y2": lat_y2,
            "bbox_lon_x2": lon_x2,
            "bbox_width_geo": bbox_width_geo,
            "bbox_height_geo": bbox_height_geo,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": conf,
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_points": json.dumps(mask_pixel_coords.tolist()),
            "mask_points_gis": json.dumps(mask_gis_coords)
        })

        masks_out.append(mask_pixel_coords)

    return object_data, masks_out

# ======================== Per-Tile Confidence Filter ========================
def maximum_likelihood_suppression(results, conf_threshold=0.5):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results

    confs = results.boxes.conf
    keep = confs >= conf_threshold

    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results

    results.boxes.data = results.boxes.data[keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[keep]
    return results

# ======================== Global Suppression with DBSCAN ========================
def global_suppression_pipeline(object_infos, eps_meters=3.0, min_samples=1):
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    df = pd.DataFrame(object_infos)
    valid_rows = []
    transformed_centroids = []

    for i, mp in enumerate(df['mask_points_gis']):
        try:
            points = json.loads(mp)
            if len(points) < 3:
                continue
            poly = Polygon(points)
            if not poly.is_valid or poly.is_empty:
                continue
            lon, lat = poly.centroid.x, poly.centroid.y
            x, y = transformer.transform(lon, lat)

            if not np.isfinite(x) or not np.isfinite(y):
                continue

            transformed_centroids.append([x, y])
            valid_rows.append(df.iloc[i])
        except Exception as e:
            continue

    if not transformed_centroids or not valid_rows:
        print("⚠️ No valid geometries found for suppression. Returning original data.")
        return object_infos

    valid_df = pd.DataFrame(valid_rows)
    clustering = DBSCAN(eps=eps_meters, min_samples=min_samples).fit(transformed_centroids)
    valid_df['cluster'] = clustering.labels_

    merged_data = []
    for cluster_id in valid_df['cluster'].unique():
        cluster_df = valid_df[valid_df['cluster'] == cluster_id]
        try:
            all_polys = [Polygon(json.loads(mp)) for mp in cluster_df['mask_points_gis'] if len(json.loads(mp)) >= 3]
            merged_poly = unary_union(all_polys)
            if merged_poly.is_empty or not merged_poly.is_valid:
                continue
        except:
            continue

        lon_c, lat_c = merged_poly.centroid.x, merged_poly.centroid.y
        minx, miny, maxx, maxy = merged_poly.bounds
        bbox_width = maxx - minx
        bbox_height = maxy - miny

        merged_data.append({
            "latitude": lat_c,
            "longitude": lon_c,
            "bbox_lon_x1": minx,
            "bbox_lat_y1": miny,
            "bbox_lon_x2": maxx,
            "bbox_lat_y2": maxy,
            "bbox_width_geo": bbox_width,
            "bbox_height_geo": bbox_height,
            "area_geo": merged_poly.area,
            "pixel_count": cluster_df['pixel_count'].sum(),
            "class": int(cluster_df['class'].mode().iloc[0]),
            "confidence": float(cluster_df['confidence'].max()),
            "tile_name": ','.join(cluster_df['tile_name'].unique()),
            "mask_points_gis": json.dumps([list(pt) for pt in merged_poly.exterior.coords])
        })

    return merged_data

# ======================== Inference Runner ========================
import threading
def run_inference(folder_path, model_path, final_csv_path,
                  window_size=3000, stride=3000, batch_size=32):

    model = YOLO(model_path)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    # GPU Warmup
    _ = model([Image.new('RGB', (window_size, window_size))], device=device, verbose=False)

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    accumulate_object_infos, accumulate_masks = [], []
    total_images = len(tif_images)

    overall_bar = tqdm(enumerate(tif_images, 1), total=total_images, desc="Overall Progress", unit="image")

    for idx, tif_image in overall_bar:
        start_time = time.time()
        slices = slide_geotiff(tif_image, window_size, stride)
        slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

        for batch in tqdm(slice_batches, desc=f"Processing {os.path.basename(tif_image)}", leave=False):
            # Preprocess batch to PIL only once
            pil_imgs_batch = [Image.fromarray(np.moveaxis(slice_data["img_array"][:3], 0, -1)) for slice_data in batch]

            with torch.inference_mode():
                results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

            for slice_data, results in zip(batch, results_batch):
                # results = maximum_likelihood_suppression(results, conf_threshold=0.5)
                object_info, masks = overlay_segmentation_on_array(
                    results,
                    transform=slice_data["transform"],
                    img_shape=pil_imgs_batch[0].size + (3,),
                    tif_image=os.path.basename(tif_image),
                    offset_xy=slice_data["offset_xy"]
                )
                accumulate_object_infos.extend(object_info)
                accumulate_masks.extend(masks)

        torch.cuda.empty_cache()
        gc.collect()
        elapsed = time.time() - start_time
        overall_bar.set_postfix({
            'Last_image_time': f"{timedelta(seconds=int(elapsed))}",
            'Images_left': total_images - idx,
        })
        # Save temp checkpoint every 100 images in background
        if idx % 100 == 0:
            temp_csv_path = f"lat_cong_csv/temp_{idx}.csv"
            threading.Thread(
                target=lambda: pd.DataFrame(accumulate_object_infos).to_csv(temp_csv_path, index=False),
                daemon=True
            ).start()
            print(f"📝 Saved checkpoint CSV in background: {temp_csv_path}")

    overall_bar.close()

    print("\n🔍 Applying Global Suppression...")
    # # suppressed_infos = global_suppression_pipeline(accumulate_object_infos)
    # pd.DataFrame(suppressed_infos).to_csv(final_csv_path, index=False)
    pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
    print(f"\n✅ Final suppressed results saved to {final_csv_path}")



In [ ]:
run_inference(
    folder_path="D:/Data/",
    model_path="C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt",
    final_csv_path="lat_cong_csv_no_mls/final.csv"
)

Using device: cuda


Overall Progress:   0%|          | 0/3480 [00:00<?, ?image/s]

Processing m_3008502_nw_16_030_20211103.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3008502_ne_16_030_20211109.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3008503_nw_16_030_20211109.tif:   0%|          | 0/2 [00:00<?, ?it/s]

...

Processing m_3408759_sw_16_030_20211106.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3408759_se_16_030_20211106.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3408760_sw_16_030_20211106.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3408760_se_16_030_20211106.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3408761_sw_16_030_20211106.tif:   0%|          | 0/2 [00:00<?, ?it/s]